In [5]:
from graphviz import Digraph

def create_panopticon_arch():
    dot = Digraph('MultiSensor_Panopticon', format='png')
    dot.attr(dpi='300', rankdir='TB', compound='true')
    dot.attr('node', fontname='Arial', shape='box', style='filled,rounded', fontsize='12')

    # --- 1. Multi-Sensor Input Stage (PE) ---
    with dot.subgraph(name='cluster_input') as c:
        c.attr(label='I. Sensor-Specific Patch Embedding', style='dashed', fontcolor='blue')
        sensors = [
            ('s2', 'Sentinel-2', '#CCEBC5'),
            ('l89', 'Landsat 8/9', '#B3CDE3'),
            ('s5p', 'TROPOMI (S5P)', '#DECBE4')
        ]
        for key, name, color in sensors:
            # 简化代码中的 Conv3d + ChnAttn + Proj
            c.node(f'PE_{key}', f'{name} PE\n(Conv3D + ChnAttn)', fillcolor=color)

    # --- 2. Shared Backbone (ViT Blocks) ---
    with dot.subgraph(name='cluster_backbone') as c:
        c.attr(label='II. Universal Backbone (DinoV2 Based)', style='filled', fillcolor='#F9F9F9')
        
        # 内部展示一个典型的 Block 结构，体现 DS-LN
        c.node('LN1', 'Domain-Specific LN 1\n(Selects s2/l89/s5p parameters)', fillcolor='#FFD1D1', color='red', penwidth='2')
        c.node('Attn', 'Shared Attention\n(Global Context)', fillcolor='#FFF2CC')
        c.node('LN2', 'Domain-Specific LN 2', fillcolor='#FFD1D1', color='red', penwidth='2')
        c.node('FFN_1', 'Domain-Specific SwiGLU FFN 1', fillcolor='#FFD1D1', color='red', penwidth='2')
        c.node('FFN_2', 'Shared SwiGLU FFN 2\n(Knowledge Base)', fillcolor='#FFF2CC')
        
        c.edge('LN1', 'Attn')
        c.edge('Attn', 'LN2')
        c.edge('LN2', 'FFN_1')
        c.edge('FFN_1', 'FFN_2')
        
        # 标注 12层 堆叠
        c.node('Stack', '... Repeat x12 Blocks ...', shape='none', style='')
        c.edge('FFN_2', 'Stack')

    # --- 3. Output Heads ---
    with dot.subgraph(name='cluster_heads') as c:
        c.attr(label='III. Task-Specific Heads', style='dashed', fontcolor='green')
        for key, name, color in sensors:
            c.node(f'Head_{key}', f'CLS Head ({name})\nMethane Yes/No', fillcolor=color)

    # --- 连接全局 ---
    for key, _, _ in sensors:
        dot.edge(f'PE_{key}', 'LN1', lhead='cluster_backbone')
        dot.edge('Stack', f'Head_{key}', ltail='cluster_backbone')

    dot.render('panopticon_final_arch', cleanup=True)
    print("架构图已生成：panopticon_final_arch.png")

if __name__ == "__main__":
    create_panopticon_arch()

架构图已生成：panopticon_final_arch.png


In [ ]:
from graphviz import Digraph

def create_no_container_arch():
    # compound=true 仍然保留，以确保整体布局严谨
    dot = Digraph('Panopticon_No_Container', format='png')
    dot.attr(dpi='300', rankdir='TB', compound='true')
    dot.attr('node', fontname='Arial', shape='box', style='filled,rounded', fontsize='12')

    # --- 1. Sensor-Specific Input (PE) ---
    with dot.subgraph(name='cluster_input') as c:
        c.attr(label='I. Sensor-Specific Patch Embedding', style='dashed', fontcolor='#444444')
        sensors = [('s2', 'Sentinel-2', '#CCEBC5'), ('l89', 'Landsat 8/9', '#B3CDE3'), ('s5p', 'S5P (TROPOMI)', '#DECBE4')]
        for key, name, color in sensors:
            c.node(f'PE_{key}', f'{name} PE\n(3D Conv + ChnAttn)', fillcolor=color)

    # --- 2. Backbone (已经去掉了内部的 5x Adapter 容器) ---
    with dot.subgraph(name='cluster_backbone') as c:
        c.attr(label='II. Hybrid ViT Backbone', style='filled', fillcolor='#F9F9F9')
        
        c.node('Blocks_Shared', '7 x Shared Transformer Blocks\n(Universal Features)', fillcolor='#FFF2CC', width='4')
        
        # 直接定义原本在容器内的节点
        # 核心分叉点 (Input X)
        dot.node('Split', '', shape='point', width='0.1')
        
        # 中心主干道
        dot.node('Shared_Core', '5 x Shared NestedTensorBlock\n(Attn + MLP)', fillcolor='#FFF2CC', penwidth='2', width='3')
        
        # 侧边旁路适配器
        dot.node('Ad_S2', 'Tiny Adapter\n(Sentinel-2)', fillcolor='#CCEBC5', style='filled,dashed')
        dot.node('Ad_L89', 'Tiny Adapters\n(Landsat-8/9)', fillcolor='#B3CDE3', style='filled,dashed')
        dot.node('Ad_S5p', 'Tiny Adapters\n(Sentinel-5p)', fillcolor='#FFD1D1', style='filled,dashed')
        
        # 核心汇合点
        dot.node('Sum', 'Sum (+)', shape='circle', fillcolor='#D1FFD1')

        # 保持横向对齐逻辑
        with dot.subgraph() as s:
            s.attr(rank='same')
            dot.node('Ad_S2')
            dot.node('Shared_Core')
            dot.node('Ad_L89')
            dot.node('Ad_S5p')

        # --- 内部连线逻辑 ---
        # 增加从 Blocks_Shared 到 Split 的权重和距离
        dot.edge('Blocks_Shared', 'Split', minlen='1.2')
        
        # 主干直线
        dot.edge('Split', 'Shared_Core', weight='10')
        dot.edge('Shared_Core', 'Sum', weight='10')
        
        # 并行分叉
        dot.edge('Split', 'Ad_S2', style='dashed')
        dot.edge('Split', 'Ad_L89', style='dashed')
        dot.edge('Split', 'Ad_S5p', style='dashed')
        
        # 汇合
        dot.edge('Ad_S2', 'Sum', style='dashed')
        dot.edge('Ad_L89', 'Sum', style='dashed')
        dot.edge('Ad_S5p', 'Sum', style='dashed')

    # --- 3. Classifier Heads ---
    with dot.subgraph(name='cluster_heads') as c:
        c.attr(label='III. Specialized CLS Heads', style='dashed')
        c.node('H_S2', 'Head (Sentinel-2)', fillcolor='#CCEBC5')
        c.node('H_L89', 'Head (Landsat 8/9)', fillcolor='#B3CDE3')
        c.node('H_S5p', 'Head (Sentinel-5p)', fillcolor='#FFD1D1')

    # --- 全局连线 ---
    for key, _, _ in sensors:
        dot.edge(f'PE_{key}', 'Blocks_Shared')
    
    # 直接连向 Head
    dot.edge('Sum', 'H_S2')
    dot.edge('Sum', 'H_L89')
    dot.edge('Sum', 'H_S5p')

    dot.render('panopticon_no_container', cleanup=True)
    print("架构图已生成：panopticon_residual_adapter.png")

if __name__ == "__main__":
    create_no_container_arch()

架构图已生成（去掉了 Adapter 容器）：panopticon_residual_adapter.png


In [ ]:
from graphviz import Digraph

def create_modified_arch():
    dot = Digraph('Panopticon_Modified', format='png')
    dot.attr(dpi='300', rankdir='TB', compound='true')
    dot.attr('node', fontname='Arial', shape='box', style='filled,rounded', fontsize='12')

    # --- 1. Sensor-Specific Input (PE) ---
    with dot.subgraph(name='cluster_input') as c:
        c.attr(label='I. Sensor-Specific Patch Embedding', style='dashed', fontcolor='#444444')
        sensors = [('s2', 'Sentinel-2', '#CCEBC5'), ('l89', 'Landsat 8/9', '#B3CDE3'), ('s5p', 'S5P (TROPOMI)', '#DECBE4')]
        for key, name, color in sensors:
            c.node(f'PE_{key}', f'{name} PE\n(3D Conv + ChnAttn)', fillcolor=color)

    # --- 2. Adapter Stage (First 5 Layers - Frozen Backbone) ---
    with dot.subgraph(name='cluster_adapters') as c:
        c.attr(label='II. Adapter Stage (First 5 Layers)\n[Backbone Frozen]', style='filled', fillcolor='#F0F0F0')
        
        # Logic nodes
        c.node('Split', 'Input Split', shape='point', width='0.1')
        c.node('Sum', 'Sum (+)', shape='circle', fillcolor='#D1FFD1')
        
        # The Frozen Backbone part
        c.node('Shared_Core', '5 x Shared NestedTensorBlock\n(FROZEN)', fillcolor='#E0E0E0', fontcolor='#777777', penwidth='2')
        
        # The Active Adapters
        c.node('Ad_S2', 'Tiny Adapter\n(Sentinel-2)', fillcolor='#CCEBC5', style='filled,dashed')
        c.node('Ad_L89', 'Tiny Adapters\n(Landsat-8/9)', fillcolor='#B3CDE3', style='filled,dashed')
        c.node('Ad_S5p', 'Tiny Adapters\n(S5P)', fillcolor='#FFD1D1', style='filled,dashed')

        # Alignment
        with c.subgraph() as s:
            s.attr(rank='same')
            s.node('Ad_S2')
            s.node('Shared_Core')
            s.node('Ad_L89')
            s.node('Ad_S5p')

        # Internal Connections
        c.edge('Split', 'Shared_Core', weight='10')
        c.edge('Shared_Core', 'Sum', weight='10')
        c.edge('Split', 'Ad_S2', style='dashed')
        c.edge('Split', 'Ad_L89', style='dashed')
        c.edge('Split', 'Ad_S5p', style='dashed')
        c.edge('Ad_S2', 'Sum', style='dashed')
        c.edge('Ad_L89', 'Sum', style='dashed')
        c.edge('Ad_S5p', 'Sum', style='dashed')

    # --- 3. Shared Backbone (Last 7 Layers) ---
    with dot.subgraph(name='cluster_shared') as c:
        c.attr(label='III. Global Feature Extraction (Last 7 Layers)', style='filled', fillcolor='#F9F9F9')
        c.node('Blocks_Shared', '7 x Shared Transformer Blocks\n(Universal Features)', fillcolor='#FFF2CC', width='4')

    # --- 4. Classifier Heads ---
    with dot.subgraph(name='cluster_heads') as c:
        c.attr(label='IV. Specialized CLS Heads', style='dashed')
        c.node('H_S2', 'Head (Sentinel-2)', fillcolor='#CCEBC5')
        c.node('H_L89', 'Head (Landsat 8/9)', fillcolor='#B3CDE3')
        c.node('H_S5p', 'Head (S5P)', fillcolor='#FFD1D1')

    # --- Global Flow ---
    for key, _, _ in sensors:
        dot.edge(f'PE_{key}', 'Split')
    
    dot.edge('Sum', 'Blocks_Shared')
    
    dot.edge('Blocks_Shared', 'H_S2')
    dot.edge('Blocks_Shared', 'H_L89')
    dot.edge('Blocks_Shared', 'H_S5p')

    dot.render('panopticon_updated_arch', cleanup=True, view=False)
    print("Graph generated: panopticon_updated_arch.png")

if __name__ == "__main__":
    create_modified_arch()

In [ ]:
from graphviz import Digraph

def create_panopticon_emit_arch():
    dot = Digraph('Panopticon_EMIT', format='png')
    dot.attr(dpi='300', rankdir='TB', compound='true')
    dot.attr('node', fontname='Arial', shape='box', style='filled,rounded', fontsize='12')

    # Color Palette
    c_s2 = '#CCEBC5'   # Green
    c_l89 = '#B3CDE3'  # Blue
    c_s5p = '#DECBE4'  # Purple
    c_emit = '#FFF9C4' # Yellow (EMIT/WV3)

    # --- 1. Sensor-Specific Input (PE) ---
    with dot.subgraph(name='cluster_input') as c:
        c.attr(label='I. Sensor-Specific Patch Embedding', style='dashed', fontcolor='#444444')
        sensors = [
            ('s2', 'Sentinel-2', c_s2), 
            ('l89', 'Landsat 8/9', c_l89), 
            ('s5p', 'S5P (TROPOMI)', c_s5p),
            ('emit', 'EMIT (Sim. WV-3)', c_emit) # New Sensor
        ]
        for key, name, color in sensors:
            c.node(f'PE_{key}', f'{name} PE\n(3D Conv + ChnAttn)', fillcolor=color)

    # --- 2. Adapter Stage (First 5 Layers) ---
    with dot.subgraph(name='cluster_adapters') as c:
        c.attr(label='II. Adapter Stage (First 5 Layers)\n[Backbone Frozen]', style='filled', fillcolor='#F0F0F0')
        
        dot.node('Split', 'Input Split', shape='point', width='0.1')
        dot.node('Sum', 'Sum (+)', shape='circle', fillcolor='#D1FFD1')
        
        # Frozen Backbone Core
        dot.node('Shared_Core', '5 x Shared NestedTensorBlock\n(FROZEN)', fillcolor='#E0E0E0', fontcolor='#777777', penwidth='2')
        
        # Adapters (Include new EMIT adapter)
        dot.node('Ad_S2', 'Tiny Adapter\n(Sentinel-2)', fillcolor=c_s2, style='filled,dashed')
        dot.node('Ad_L89', 'Tiny Adapter\n(Landsat-8/9)', fillcolor=c_l89, style='filled,dashed')
        dot.node('Ad_S5p', 'Tiny Adapter\n(S5P)', fillcolor=c_s5p, style='filled,dashed')
        dot.node('Ad_EMIT', 'Tiny Adapter\n(EMIT/WV-3)', fillcolor=c_emit, style='filled,dashed') # New Adapter

        # Horizontal Alignment
        with dot.subgraph() as s:
            s.attr(rank='same')
            dot.node('Ad_S2')
            dot.node('Ad_L89')
            dot.node('Shared_Core')
            dot.node('Ad_S5p')
            dot.node('Ad_EMIT')

        # Logic Flow
        dot.edge('Split', 'Shared_Core', weight='10')
        dot.edge('Shared_Core', 'Sum', weight='10')
        
        for key in ['S2', 'L89', 'S5p', 'EMIT']:
            dot.edge('Split', f'Ad_{key}', style='dashed')
            dot.edge(f'Ad_{key}', 'Sum', style='dashed')

    # --- 3. Shared Backbone (Last 7 Layers) ---
    with dot.subgraph(name='cluster_shared') as c:
        c.attr(label='III. Global Feature Extraction (Last 7 Layers)', style='filled', fillcolor='#F9F9F9')
        dot.node('Blocks_Shared', '7 x Shared Transformer Blocks\n(Universal Features)', fillcolor='#FFF2CC', width='5')

    # --- 4. Specialized Heads ---
    with dot.subgraph(name='cluster_heads') as c:
        c.attr(label='IV. Specialized CLS Heads', style='dashed')
        dot.node('H_S2', 'Head (Sentinel-2)', fillcolor=c_s2)
        dot.node('H_L89', 'Head (Landsat 8/9)', fillcolor=c_l89)
        dot.node('H_S5p', 'Head (S5P)', fillcolor=c_s5p)
        dot.node('H_EMIT', 'Head (EMIT/WV-3)', fillcolor=c_emit) # New Head

    # --- Global Connections ---
    for key, _, _ in sensors:
        dot.edge(f'PE_{key}', 'Split')
    
    dot.edge('Sum', 'Blocks_Shared')
    
    for key in ['S2', 'L89', 'S5p', 'EMIT']:
        dot.edge('Blocks_Shared', f'H_{key}')

    dot.render('panopticon_emit_wv3', cleanup=True)
    print("Graph updated with EMIT/WV-3 nodes.")

if __name__ == "__main__":
    create_panopticon_emit_arch()

Graph updated with EMIT/WV-3 nodes.


In [23]:
from graphviz import Digraph

def draw_balanced_adapter_block():
    dot = Digraph('SensorAdapterBlock_Balanced', format='png')
    dot.attr(dpi='300', rankdir='LR', compound='true')
    dot.attr('node', fontname='Arial', shape='box', style='filled,rounded', fontsize='12')

    # --- 输入节点：保持精简尺寸 ---
    dot.node('input', 'Input Feature\n(768-dim)', shape='circle', 
             fillcolor='#E1E1E1', width='0.8', height='0.8', fixedsize='true', fontsize='10')

    # --- 1. 主路径 (Shared Block) ---
    with dot.subgraph(name='cluster_main') as c:
        c.attr(label='Shared Block', style='filled', fillcolor='#F0F0F0')
        # block_core 现在会被 minlen 推向右侧
        c.node('block_core', 'Standard NestedTensorBlock\n(Attn + MLP + Internal Residuals)', fillcolor='#FFF2CC')

    # --- 2. 适配器路径 (Parallel Adapters) ---
    with dot.subgraph(name='cluster_adapters') as c:
        c.attr(label='Trainable Sensor Adapters', style='dashed', fontcolor='#CC0000')
        
        with dot.subgraph(name='cluster_bottleneck') as b:
            b.attr(label='TinyResidualAdapter Structure', style='dotted')
            b.node('down', 'Linear Down-proj\n(768 -> 16)', fillcolor='#FFD1D1')
            b.node('act', 'GELU Activation', shape='ellipse', fillcolor='#FFD1D1')
            b.node('up', 'Linear Up-proj\n(16 -> 768)', fillcolor='#FFD1D1')
            b.edge('down', 'act')
            b.edge('act', 'up')

    # --- 3. 融合阶段 ---
    dot.node('plus', 'Sum (+)', shape='circle', fillcolor='#D1FFD1', 
             width='0.5', height='0.5', fixedsize='true', fontsize='10')
    dot.node('output', 'Output Feature\n(768-dim)', shape='circle', 
             fillcolor='#E1E1E1', width='0.8', height='0.8', fixedsize='true', fontsize='10')

    # --- 关键连线调整 ---
    # 为主路径增加 minlen，使其向右移动，与旁路的 down 节点对齐起始位置或在视觉上更居中
    dot.edge('input', 'block_core', minlen='2.0') 
    
    # 旁路连线保持默认或微调
    dot.edge('input', 'down', label='Select via ModuleDict\n(e.g., S2, L8/9 or S5p)', fontsize='9')
    
    dot.edge('block_core', 'plus')
    dot.edge('up', 'plus')
    dot.edge('plus', 'output')

    dot.render('adapter_balanced_layout', cleanup=True)
    print("结构图已生成（Shared Block 已向右移动）：adapter_balanced_layout.png")

if __name__ == "__main__":
    draw_balanced_adapter_block()

结构图已生成（Shared Block 已向右移动）：adapter_balanced_layout.png


In [1]:
from graphviz import Digraph


def create_anysensor_methane_arch():
    dot = Digraph('AnySensor_Methane_Detector', format='png')
    dot.attr(dpi='300', rankdir='TB', compound='true', splines='ortho')
    dot.attr(
        'node',
        fontname='Arial',
        shape='box',
        style='filled,rounded',
        fontsize='12',
        color='#666666'
    )
    dot.attr('edge', color='#666666', arrowsize='0.8')

    # Color palette
    c_s2 = '#CCEBC5'     # green
    c_l89 = '#B3CDE3'    # blue
    c_s5p = '#DECBE4'    # purple
    c_emit = '#FFF9C4'   # yellow
    c_shared = '#EAEAEA'
    c_feature = '#FFF2CC'
    c_head = '#FFD6A5'

    sensors = [
        ('s2', 'Sentinel-2', c_s2),
        ('l89', 'Landsat 8/9', c_l89),
        ('s5p', 'S5P (TROPOMI)', c_s5p),
        ('emit', 'EMIT (Sim. WV-3)', c_emit),
    ]

    # -------------------------------
    # I. Sensor-specific patch embedding
    # -------------------------------
    with dot.subgraph(name='cluster_input') as c:
        c.attr(
            label='I. Sensor-Specific Patch Embedding',
            style='dashed',
            color='#999999',
            fontcolor='#444444'
        )
        for key, name, color in sensors:
            c.node(
                f'PE_{key}',
                f'{name}\nPatch Embedding\n(3D Conv + Channel Attention)',
                fillcolor=color
            )

    # -------------------------------
    # II. Adapter stage
    # -------------------------------
    with dot.subgraph(name='cluster_adapters') as c:
        c.attr(
            label='II. Sensor-Aware Adapter Stage (First 5 Layers)\n[Backbone Frozen]',
            style='filled',
            fillcolor='#F5F5F5',
            color='#CCCCCC',
            fontcolor='#444444'
        )

        c.node('Split', '', shape='point', width='0.12', fillcolor='#666666')
        c.node('Sum', '+', shape='circle', width='0.5', fillcolor='#D9F2D9')

        c.node(
            'Shared_Core',
            '5 × Shared Transformer Blocks\n(Frozen Backbone)',
            fillcolor=c_shared,
            fontcolor='#666666',
            penwidth='2'
        )

        c.node('Ad_S2', 'Tiny Adapter\n(Sentinel-2)', fillcolor=c_s2, style='filled,dashed')
        c.node('Ad_L89', 'Tiny Adapter\n(Landsat 8/9)', fillcolor=c_l89, style='filled,dashed')
        c.node('Ad_S5p', 'Tiny Adapter\n(S5P)', fillcolor=c_s5p, style='filled,dashed')
        c.node('Ad_EMIT', 'Tiny Adapter\n(EMIT / WV-3)', fillcolor=c_emit, style='filled,dashed')

        # same-rank alignment
        with c.subgraph() as s:
            s.attr(rank='same')
            s.node('Ad_S2')
            s.node('Ad_L89')
            s.node('Shared_Core')
            s.node('Ad_S5p')
            s.node('Ad_EMIT')

    # -------------------------------
    # III. Shared feature extraction
    # -------------------------------
    with dot.subgraph(name='cluster_shared') as c:
        c.attr(
            label='III. Global Feature Extraction (Last 7 Layers)',
            style='filled',
            fillcolor='#FAFAFA',
            color='#DDDDDD',
            fontcolor='#444444'
        )
        c.node(
            'Blocks_Shared',
            '7 × Shared Transformer Blocks\n(Universal Features)',
            fillcolor=c_feature,
            width='5'
        )

    # -------------------------------
    # IV. Unified head
    # -------------------------------
    with dot.subgraph(name='cluster_head') as c:
        c.attr(
            label='IV. Unified Prediction Head',
            style='dashed',
            color='#999999',
            fontcolor='#444444'
        )
        c.node(
            'H_overall',
            'Overall Methane Detection Head\n(Sensor-Agnostic Classifier)',
            fillcolor=c_head,
            width='3'
        )

    # -------------------------------
    # Connections: input to split
    # -------------------------------
    for key, _, _ in sensors:
        dot.edge(f'PE_{key}', 'Split')

    # Main shared branch
    dot.edge('Split', 'Shared_Core', weight='10')
    dot.edge('Shared_Core', 'Sum', weight='10')

    # Adapter residual branches
    adapter_map = {
        's2': 'Ad_S2',
        'l89': 'Ad_L89',
        's5p': 'Ad_S5p',
        'emit': 'Ad_EMIT',
    }

    for key in adapter_map:
        dot.edge('Split', adapter_map[key], style='dashed')
        dot.edge(adapter_map[key], 'Sum', style='dashed')

    # Shared downstream path
    dot.edge('Sum', 'Blocks_Shared')
    dot.edge('Blocks_Shared', 'H_overall')

    # Render
    output_path = dot.render('anysensor_methane_detector_arch', cleanup=True)
    print(f'Graph saved to: {output_path}')


if __name__ == "__main__":
    create_anysensor_methane_arch()

Graph saved to: anysensor_methane_detector_arch.png


In [ ]:
from graphviz import Digraph


def create_multisensor_arch():
    dot = Digraph('AnySensor_Methane_Model', format='png')
    dot.attr(dpi='300', rankdir='TB', compound='true')
    dot.attr('node', fontname='Arial', shape='box', style='filled,rounded', fontsize='12')

    # -------------------------
    # 1. Multi-Sensor Input
    # -------------------------
    with dot.subgraph(name='cluster_input') as c:
        c.attr(label='I. Sensor-Specific Patch Embedding', style='dashed', fontcolor='blue')

        sensors = [
            ('s2', 'Sentinel-2', '#CCEBC5'),
            ('l89', 'Landsat 8/9', '#B3CDE3'),
            ('s5p', 'TROPOMI (S5P)', '#DECBE4'),
            ('wv3', 'WorldView-3', '#FFF2CC')
        ]

        for key, name, color in sensors:
            c.node(
                f'PE_{key}',
                f'{name} PE\n(Conv3D + Channel Attention)',
                fillcolor=color
            )

    # -------------------------
    # 2. Shared Backbone
    # -------------------------
    with dot.subgraph(name='cluster_backbone') as c:
        c.attr(label='II. Universal Transformer Backbone', style='filled', fillcolor='#F9F9F9')

        c.node(
            'LN1',
            'Domain-Specific LN 1\n(selects s2/l89/s5p/wv3 params)',
            fillcolor='#FFD1D1',
            color='red',
            penwidth='2'
        )

        c.node(
            'Attn',
            'Shared Attention\n(Global Context)',
            fillcolor='#FFF2CC'
        )

        c.node(
            'LN2',
            'Domain-Specific LN 2',
            fillcolor='#FFD1D1',
            color='red',
            penwidth='2'
        )

        c.node(
            'FFN_1',
            'Domain-Specific SwiGLU FFN 1',
            fillcolor='#FFD1D1',
            color='red',
            penwidth='2'
        )

        c.node(
            'FFN_2',
            'Shared SwiGLU FFN 2\n(Knowledge Base)',
            fillcolor='#FFF2CC'
        )

        c.edge('LN1', 'Attn')
        c.edge('Attn', 'LN2')
        c.edge('LN2', 'FFN_1')
        c.edge('FFN_1', 'FFN_2')

        c.node('Stack', '... Repeat ×12 Transformer Blocks ...', shape='none', style='')
        c.edge('FFN_2', 'Stack')

    # -------------------------
    # 3. Unified Output Head
    # -------------------------
    with dot.subgraph(name='cluster_heads') as c:
        c.attr(label='III. Unified Prediction Head', style='dashed', fontcolor='green')

        c.node(
            'Head',
            'Unified Methane Detection Head\n(Binary Classification)',
            fillcolor='#FFD6A5',
            width='3'
        )

    # -------------------------
    # Global Connections
    # -------------------------
    for key, _, _ in sensors:
        dot.edge(f'PE_{key}', 'LN1', lhead='cluster_backbone')

    dot.edge('Stack', 'Head', ltail='cluster_backbone')

    dot.render('anysensor_transformer_arch', cleanup=True)
    print("架构图已生成：anysensor_transformer_arch.png")


if __name__ == "__main__":
    create_multisensor_arch()